# 수정본의 실행·해석 범위

이 노트북은 2026-10-07 코드 수정본입니다. 저장된 출력과 과거 성능 숫자는 제거했습니다. 실제 Excel 전체 처리, 모델 학습·탐색과 성능 재평가는 이번 수정에서 실행하지 않았습니다. 노트북 폴더를 작업 디렉터리로 사용하고 원자료와 `ml_validation.py`, 프로젝트 의존성을 준비한 뒤 위에서 아래로 실행해야 합니다.

`수익률`은 기존 Excel 값을 신뢰하지 않고 매수가·매도가·구매주식수·투자원금으로 다시 계산합니다. 수수료·세금·배당을 포함하지 않은 계산입니다. 이전에 열람한 2021년 이후 자료는 **과거 자료 재평가**이며, 새로 봉인된 최종 테스트가 아닙니다. 전년도 재무제표라는 이유만으로 매수 당시 공개된 정보라고 보장할 수 없으므로 공시 시점 자료의 연결은 별도 과제입니다. 입력 계약과 한계는 `NOTEBOOK-REPAIR-NOTES.md`를 확인하세요.

현재 보관된 `ALL_profit_final(개).xlsx`의 13,979행 중 3행은 매도가가 없습니다(Excel 행 942, 952, 2536). 이 파일을 그대로 읽으면 `recalculate_returns`가 의도적으로 오류를 냅니다. 결측을 0으로 바꾸거나 행을 자동 삭제해 학습을 진행하지 않습니다. 원가격·보유기간 자료를 확인해 처리 근거를 정하기 전에는 전체 학습을 재개할 수 없습니다. 초기 회계 진단 비교에서 유효 13,976행의 포지션 손익 부호 141개가 양수에서 비양수로 바뀌었습니다. 이 수치는 새 가격상승 정답의 변경 개수와 다릅니다. 이는 모델 성능 검증이 아닙니다. 근거: `saved-label-audit.json`.

## 분류 목표를 가격 상승으로 분리했습니다

현재 목표는 **수정주가 기준 가격 상승 여부**입니다. `add_price_direction_targets(recalculate_returns(frame))`로 회계 열을 유지한 뒤 `가격변화율 = 100 × (매도가 / 매수가 − 1)`, `가격상승여부 = int(매도가 > 매수가)`를 추가합니다. 모델의 정답 y는 `가격상승여부`이고, `수익률_양음`은 배분된 원금·정수 주식 수에 따른 회계 손익 부호로 보존합니다.

원자료의 0주 행 25개 중 가격이 상승한 8행은 포지션 이익 0이지만 가격상승 정답은 1입니다(`verification/zero-share-audit.json`). 수정주가는 당시 체결 가능한 실제 호가와 같다고 보장되지 않습니다. 당시 가격 없이 수정주가로 정수 주식 수를 계산한 백테스트의 현실성, 원자료의 주식분할·배당 등 보정 정의는 아직 확인하지 않았습니다. 따라서 새 목표는 기업 자체의 성장·투자 수익의 증명이 아니라 저장된 수정주가의 기간 변화입니다. 기존 포지션 이익 분류의 성능 숫자를 새 목표의 성능으로 재사용하지 않습니다.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import pandas as pd
import KSE_가치팩터_functions as kf
import matplotlib.pyplot as plt
from ml_validation import recalculate_returns, temporal_split, add_price_direction_targets

In [ ]:
# 엑셀 파일 경로 (파일 경로는 실제 경로로 수정해야 합니다)
file_path = 'KSE_FIN_DATA_2023.xlsx'

# 전체 데이터셋 로드 (파일 경로와 시트 이름을 실제 데이터에 맞게 수정하세요)
df_close = pd.read_excel(file_path, sheet_name='수정주가')

# 날짜 열을 datetime 형태로 변환
df_close['Symbol Name'] = pd.to_datetime(df_close['Symbol Name'])

close_1 = df_close.copy()

# 첫 번째 열을 인덱스로 설정
close_1.set_index(close_1.columns[0], inplace=True)

# 인덱스를 날짜 형식으로 변환
close_1.index = pd.to_datetime(close_1.index)

In [ ]:
### PER

In [ ]:
# 데이터 전처리

excel_file_path = 'PER_profit_final.xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))

# 분류 정답은 수정주가가 상승했는지입니다. 포지션 손익/수익률 열은 별도로 유지합니다.

# 독립변수와 종속변수 분리
X = excel_data[['1/PER', '1/PER Rank']]
y = excel_data['가격상승여부']

# 데이터를 시간에 따라 나누기 (훈련, 검증, 테스트)
train_data, validation_data, test_data = temporal_split(excel_data)

# 훈련 데이터셋으로 모델 학습
X_train = train_data[['1/PER', '1/PER Rank']]
y_train = train_data['가격상승여부']
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 테스트 데이터셋에 대한 예측
X_test = test_data[['1/PER', '1/PER Rank']]
test_data['예측_가격상승여부'] = clf.predict(X_test)

# 2021년도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_companies_2021 = test_data[(test_data['연도'] == 2021) & (test_data['예측_가격상승여부'] == 1)].sort_values('1/PER', ascending=False).head(20)

top_20_companies_2021[['연도', '기업명', '1/PER', '1/PER Rank', '가격상승여부', '예측_가격상승여부']]

In [ ]:
# 가격상승여부는 매도 시점 수정주가가 매수 시점보다 높은지를 나타냅니다.
# 예측_가격상승여부는 모델의 분류값입니다. 회계 손익 부호 수익률_양음과 다릅니다.
# 상승으로 분류한 기업을 1/PER 순서로 고르며, 상승 폭이나 확률 순위는 아닙니다.


In [ ]:
### 손으로 직접 엑셀파일을 만들었습니다. (개선필요)

In [ ]:
# Load the Excel file for backtesting
excel_backtesting_file = '머신러닝_PER_2021_백테스팅.xlsx'

# Load the two sheets from the Excel file
portfolio_1 = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_backtesting_file, sheet_name=0)))
portfolio_2 = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_backtesting_file, sheet_name=1)))

# Portfolio 1 Total Return
total_return_portfolio_1 = portfolio_1['수익'].sum()

# Portfolio 2 Total Return
total_return_portfolio_2 = portfolio_2['수익'].sum()

total_return_portfolio_1, total_return_portfolio_2

2017~2020년 검증 자료와 2021년 이후 과거 테스트 자료를 합치지 않습니다. 아래 종목 목록은 과거 테스트 구간에 대한 탐색용 결과이며 새 데이터에서의 투자 성과 증명이 아닙니다.

In [ ]:
# Load the provided Excel file
excel_file_path = 'PER_profit_final.xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))

# 분류 정답은 수정주가가 상승했는지입니다. 포지션 손익/수익률 열은 별도로 유지합니다.

# 독립변수와 종속변수 분리
X = excel_data[['1/PER', '1/PER Rank']]
y = excel_data['가격상승여부']

# 데이터를 시간에 따라 나누기 (훈련, 검증, 테스트)
train_data, validation_data, test_data = temporal_split(excel_data)

# 훈련 데이터셋으로 모델 학습
X_train = train_data[['1/PER', '1/PER Rank']]
y_train = train_data['가격상승여부']
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 검증(2017~2020)과 과거 테스트(2021~)는 별도로 유지합니다.

# 테스트 데이터셋에 대한 예측
X_test = test_data[['1/PER', '1/PER Rank']]
test_data['예측_가격상승여부'] = clf.predict(X_test)

# 모든 연도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_per_year = test_data[test_data['예측_가격상승여부'] == 1].groupby('연도').apply(lambda x: x.nlargest(20, '1/PER')).reset_index(drop=True)

top_20_per_year[['연도', '기업명', '1/PER', '1/PER Rank', '가격상승여부', '예측_가격상승여부']]

In [ ]:
# 의사결정트리 시각화
plt.figure(figsize=(20, 10))
plot_tree(clf, filled=True, feature_names=['1/PER', '1/PER Rank'], class_names=['Not rising', 'Rising'])
plt.show()

In [ ]:
# 새로운 형태로 데이터 프레임 변환
top_20_per_year = top_20_per_year[['기업명', '1/PER Rank', '연도']]
top_20_per_year.columns = ['Company', 'Rank', 'Year']

top_20_per_year  # 결과 확인

In [ ]:
output_filename = "top_20_per_year_tree.xlsx"
top_20_per_year.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")

In [ ]:
## 단순화한 모델

In [ ]:
# 트리 깊이를 3으로 제한하여 모델을 단순화
simplified_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
simplified_clf.fit(X_train, y_train)

# 의사결정트리 시각화
plt.figure(figsize=(20, 10))
simplified_tree = plot_tree(simplified_clf, filled=True, feature_names=['1/PER', '1/PER Rank'], class_names=['Not rising', 'Rising'])
plt.show()

In [ ]:
## 단순화 모델과 기존 모델의 비교(참고로 실제 분류는 기존 모델을 통하여 이루어짐)

In [ ]:
# 검증 데이터셋의 독립변수와 종속변수를 분리합니다.
X_validation = validation_data[['1/PER', '1/PER Rank']]
y_validation = validation_data['가격상승여부']

# 원본 모델을 검증 데이터셋에 적용하여 예측을 수행합니다.
original_predictions_validation = clf.predict(X_validation)

# 단순화된 모델을 검증 데이터셋에 적용하여 예측을 수행합니다.
simplified_predictions_validation = simplified_clf.predict(X_validation)

# 검증 데이터셋에 대한 원본 모델과 단순화된 모델의 정확도를 비교합니다.
original_accuracy = accuracy_score(y_validation, original_predictions_validation)
simplified_accuracy = accuracy_score(y_validation, simplified_predictions_validation)

print(f'원본 모델 정확도: {original_accuracy}')
print(f'단순화된 모델 정확도: {simplified_accuracy}')

In [ ]:
# 검증 데이터셋의 예측값을 이용하여 혼동 행렬 생성
original_conf_matrix = confusion_matrix(y_validation, original_predictions_validation)
simplified_conf_matrix = confusion_matrix(y_validation, simplified_predictions_validation)

print('원본 모델의 혼동 행렬:')
print(original_conf_matrix)

print('\n단순화된 모델의 혼동 행렬:')
print(simplified_conf_matrix)

혼동행렬은 행이 실제값, 열이 예측값일 때 `[[TN, FP], [FN, TP]]`입니다. 수정 전 저장된 숫자와 우열 해석은 제거했습니다. 재계산한 정답으로 다시 실행하기 전에는 어느 모델이 더 낫다고 결론내릴 수 없습니다. 이 노트북은 여러 기준모델을 탐색하는 구성으로, 결과를 본 뒤 최종 모델을 고르려면 새 평가 구간이 필요합니다.

In [ ]:
### PBR

In [ ]:
# 데이터 전처리

excel_file_path = 'PBR_profit_final.xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))

# 분류 정답은 수정주가가 상승했는지입니다. 포지션 손익/수익률 열은 별도로 유지합니다.

# 독립변수와 종속변수 분리
X = excel_data[['1/PBR', '1/PBR Rank']]
y = excel_data['가격상승여부']

# 데이터를 시간에 따라 나누기 (훈련, 검증, 테스트)
train_data, validation_data, test_data = temporal_split(excel_data)

# 훈련 데이터셋으로 모델 학습
X_train = train_data[['1/PBR', '1/PBR Rank']]
y_train = train_data['가격상승여부']
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 테스트 데이터셋에 대한 예측
X_test = test_data[['1/PBR', '1/PBR Rank']]
test_data['예측_가격상승여부'] = clf.predict(X_test)

# 2021년도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_companies_2021 = test_data[(test_data['연도'] == 2021) & (test_data['예측_가격상승여부'] == 1)].sort_values('1/PBR', ascending=False).head(20)

top_20_companies_2021[['연도', '기업명', '1/PBR', '1/PBR Rank', '가격상승여부', '예측_가격상승여부']]

In [ ]:
# Load the Excel file for backtesting
excel_backtesting_file = '머신러닝_PBR_2021_백테스팅.xlsx'

# Load the two sheets from the Excel file
portfolio_1 = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_backtesting_file, sheet_name=0)))
portfolio_2 = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_backtesting_file, sheet_name=1)))

# Portfolio 1 Total Return
total_return_portfolio_1 = portfolio_1['수익'].sum()

# Portfolio 2 Total Return
total_return_portfolio_2 = portfolio_2['수익'].sum()

total_return_portfolio_1, total_return_portfolio_2

In [ ]:
# PBR 파일의 입력·학습·예측·표시 열을 모두 PBR로 통일합니다.
# 데이터 전처리

excel_file_path = 'PBR_profit_final.xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))

# 분류 정답은 수정주가가 상승했는지입니다. 포지션 손익/수익률 열은 별도로 유지합니다.

# 독립변수와 종속변수 분리
X = excel_data[['1/PBR', '1/PBR Rank']]
y = excel_data['가격상승여부']

# 데이터를 시간에 따라 나누기 (훈련, 검증, 테스트)
train_data, validation_data, test_data = temporal_split(excel_data)

# 훈련 데이터셋으로 모델 학습
X_train = train_data[['1/PBR', '1/PBR Rank']]
y_train = train_data['가격상승여부']
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 테스트 데이터셋에 대한 예측
X_test = test_data[['1/PBR', '1/PBR Rank']]
test_data['예측_가격상승여부'] = clf.predict(X_test)

# 2021년도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_companies_2021 = test_data[(test_data['연도'] == 2021) & (test_data['예측_가격상승여부'] == 1)].sort_values('1/PBR', ascending=False).head(20)

top_20_companies_2021[['연도', '기업명', '1/PBR', '1/PBR Rank', '가격상승여부', '예측_가격상승여부']]

In [ ]:
### ALL모델링

In [ ]:
# 데이터 전처리

excel_file_path = 'ALL_profit_final(개).xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))

# 분류 정답은 수정주가가 상승했는지입니다. 포지션 손익/수익률 열은 별도로 유지합니다.

# 독립변수와 종속변수 분리
X = excel_data[['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank','1/PSR Rank','1/PCR Rank']]
y = excel_data['가격상승여부']

# 데이터를 시간에 따라 나누기 (훈련, 검증, 테스트)
train_data, validation_data, test_data = temporal_split(excel_data)

# 훈련 데이터셋으로 모델 학습
X_train = train_data[['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank','1/PSR Rank','1/PCR Rank']]
y_train = train_data['가격상승여부']
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 검증(2017~2020)과 과거 테스트(2021~)는 별도로 유지합니다.

# 테스트 데이터셋에 대한 예측
X_test = test_data[['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank','1/PSR Rank','1/PCR Rank']]
test_data['예측_가격상승여부'] = clf.predict(X_test)

# 2021년도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_companies_2021 = test_data[(test_data['연도'] == 2021) & (test_data['예측_가격상승여부'] == 1)].sort_values('ALL', ascending=False).head(20)


# 모든 연도 데이터에서 가격 상승으로 분류한 기업 중 가치 지표가 높은 최대 20개 기업 추출
top_20_all_year = test_data[test_data['예측_가격상승여부'] == 1].groupby('연도').apply(lambda x: x.nlargest(20, 'ALL')).reset_index(drop=True)

top_20_all_year[['연도', '기업명', '1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank',
                 '1/PSR Rank','1/PCR Rank', '가격상승여부', '예측_가격상승여부']]

In [ ]:
# 의사결정트리 시각화
plt.figure(figsize=(20, 10))
plot_tree(clf, filled=True, feature_names=['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank','1/PSR Rank','1/PCR Rank'], class_names=['Not rising', 'Rising'])
plt.show()

In [ ]:
# 새로운 형태로 데이터 프레임 변환
top_20_all_year = top_20_all_year[['기업명', 'ALL Rank', '연도']]
top_20_all_year.columns = ['Company', 'Rank', 'Year']

top_20_all_year  # 결과 확인

In [ ]:
# 트리 깊이를 3으로 제한하여 모델을 단순화
simplified_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
simplified_clf.fit(X_train, y_train)

# 의사결정트리 시각화
plt.figure(figsize=(20, 10))
simplified_tree = plot_tree(simplified_clf, filled=True, feature_names=['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank',
                                                                        '1/PBR Rank','1/PSR Rank','1/PCR Rank'], class_names=['Not rising', 'Rising'])
plt.show()

In [ ]:
# 검증 데이터셋의 독립변수와 종속변수를 분리합니다.
X_validation = validation_data[['1/PER', '1/PBR','1/PSR','1/PCR','ALL','ALL Rank','1/PER Rank','1/PBR Rank','1/PSR Rank','1/PCR Rank']]
y_validation = validation_data['가격상승여부']

# 원본 모델을 검증 데이터셋에 적용하여 예측을 수행합니다.
original_predictions_validation = clf.predict(X_validation)

# 단순화된 모델을 검증 데이터셋에 적용하여 예측을 수행합니다.
simplified_predictions_validation = simplified_clf.predict(X_validation)

# 검증 데이터셋에 대한 원본 모델과 단순화된 모델의 정확도를 비교합니다.
original_accuracy = accuracy_score(y_validation, original_predictions_validation)
simplified_accuracy = accuracy_score(y_validation, simplified_predictions_validation)

print(f'원본 모델 정확도: {original_accuracy}')
print(f'단순화된 모델 정확도: {simplified_accuracy}')

# 검증 데이터셋의 예측값을 이용하여 혼동 행렬 생성
original_conf_matrix = confusion_matrix(y_validation, original_predictions_validation)
simplified_conf_matrix = confusion_matrix(y_validation, simplified_predictions_validation)

print('원본 모델의 혼동 행렬:')
print(original_conf_matrix)

print('\n단순화된 모델의 혼동 행렬:')
print(simplified_conf_matrix)

In [ ]:
output_filename = "top_20_all_year_tree(개).xlsx"
top_20_all_year.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")